In [1]:
# ============================================================
# INDUSTRIAL HAZARD INTELLIGENCE SYSTEM
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

from ultralytics import YOLO

import cv2
import numpy as np
import time

from pathlib import Path

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path("../")

MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "ppe_yolo11m_best.pt"

INPUT_VIDEO = PROJECT_ROOT / "videos" / "input" / "input.mp4"

OUTPUT_VIDEO = PROJECT_ROOT / "videos" / "output" / "04_hazard_intelligence_demo.mp4"

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

print("=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)

# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {INPUT_VIDEO}")

# ============================================================
# VIDEO PROPERTIES
# ============================================================

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Resolution : {frame_width} x {frame_height}")
print(f"FPS        : {fps}")

# ============================================================
# OUTPUT VIDEO WRITER
# ============================================================

OUTPUT_VIDEO.parent.mkdir(
    parents=True,
    exist_ok=True
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    fps,
    (frame_width, frame_height)
)

# ============================================================
# CLASS NAMES
# ============================================================

CLASS_NAMES = model.names

# ============================================================
# COLORS
# ============================================================

SAFE_COLOR = (0, 255, 0)

WARNING_COLOR = (0, 165, 255)

DANGER_COLOR = (0, 0, 255)

CRITICAL_COLOR = (0, 0, 180)

MACHINERY_COLOR = (255, 0, 255)

ZONE_COLOR = (0, 0, 255)

# ============================================================
# TRACK HISTORY
# ============================================================

track_history = {}

# ============================================================
# PERFORMANCE VARIABLES
# ============================================================

prev_time = 0

frame_count = 0

# ============================================================
# IOU FUNCTION
# ============================================================

def calculate_iou(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])

    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)

    if interArea == 0:
        return 0

    boxAArea = (
        (boxA[2] - boxA[0]) *
        (boxA[3] - boxA[1])
    )

    boxBArea = (
        (boxB[2] - boxB[0]) *
        (boxB[3] - boxB[1])
    )

    iou = interArea / float(
        boxAArea + boxBArea - interArea
    )

    return iou

# ============================================================
# DANGER ZONE FUNCTION
# ============================================================

def create_danger_zone(box, padding=120):

    x1, y1, x2, y2 = box

    return [

        max(0, x1 - padding),

        max(0, y1 - padding),

        min(frame_width, x2 + padding),

        min(frame_height, y2 + padding)
    ]

# ============================================================
# INFERENCE LOOP
# ============================================================

print("\nStarting industrial hazard intelligence...\n")

while True:

    success, frame = cap.read()

    if not success:
        print("\nVideo processing completed.")
        break

    # --------------------------------------------------------
    # YOLO + BYTE TRACKING
    # --------------------------------------------------------

    results = model.track(

        source=frame,

        persist=True,

        tracker="bytetrack.yaml",

        conf=0.35,

        imgsz=960,

        verbose=False,

        device=0
    )

    result = results[0]

    annotated_frame = frame.copy()

    # ========================================================
    # STORE DETECTIONS
    # ========================================================

    persons = []

    helmets = []

    vests = []

    machinery = []

    # ========================================================
    # EXTRACT DETECTIONS
    # ========================================================

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()

        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        confidences = result.boxes.conf.cpu().numpy()

        track_ids = result.boxes.id.cpu().numpy().astype(int)

        for box, class_id, conf, track_id in zip(
            boxes,
            class_ids,
            confidences,
            track_ids
        ):

            class_name = CLASS_NAMES[class_id]

            x1, y1, x2, y2 = map(int, box)

            detection = {
                "box": [x1, y1, x2, y2],
                "conf": conf,
                "track_id": track_id
            }

            if class_name == "Person":
                persons.append(detection)

            elif class_name == "Hardhat":
                helmets.append(detection)

            elif class_name == "Safety Vest":
                vests.append(detection)

            elif class_name == "machinery":
                machinery.append(detection)

    # ========================================================
    # DRAW MACHINERY + DANGER ZONES
    # ========================================================

    danger_zones = []

    for machine in machinery:

        mx1, my1, mx2, my2 = machine["box"]

        # ----------------------------------------------------
        # MACHINERY BOX
        # ----------------------------------------------------

        cv2.rectangle(
            annotated_frame,
            (mx1, my1),
            (mx2, my2),
            MACHINERY_COLOR,
            3
        )

        cv2.putText(
            annotated_frame,
            "MACHINERY",
            (mx1, my1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            MACHINERY_COLOR,
            2
        )

        # ----------------------------------------------------
        # CREATE DANGER ZONE
        # ----------------------------------------------------

        zone = create_danger_zone(
            machine["box"],
            padding=150
        )

        danger_zones.append(zone)

        zx1, zy1, zx2, zy2 = zone

        # ----------------------------------------------------
        # DRAW DANGER ZONE
        # ----------------------------------------------------

        overlay = annotated_frame.copy()

        cv2.rectangle(
            overlay,
            (zx1, zy1),
            (zx2, zy2),
            ZONE_COLOR,
            -1
        )

        alpha = 0.15

        cv2.addWeighted(
            overlay,
            alpha,
            annotated_frame,
            1 - alpha,
            0,
            annotated_frame
        )

        cv2.rectangle(
            annotated_frame,
            (zx1, zy1),
            (zx2, zy2),
            ZONE_COLOR,
            2
        )

        cv2.putText(
            annotated_frame,
            "DANGER ZONE",
            (zx1, zy1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            ZONE_COLOR,
            2
        )

    # ========================================================
    # PPE + HAZARD ANALYSIS
    # ========================================================

    for person in persons:

        px1, py1, px2, py2 = person["box"]

        track_id = person["track_id"]

        # ----------------------------------------------------
        # BODY REGIONS
        # ----------------------------------------------------

        person_height = py2 - py1

        head_region = [

            px1,
            py1,
            px2,
            py1 + int(person_height * 0.30)
        ]

        torso_region = [

            px1,
            py1 + int(person_height * 0.30),
            px2,
            py1 + int(person_height * 0.70)
        ]

        # ----------------------------------------------------
        # PPE FLAGS
        # ----------------------------------------------------

        has_helmet = False

        has_vest = False

        # ----------------------------------------------------
        # HELMET CHECK
        # ----------------------------------------------------

        for helmet in helmets:

            iou = calculate_iou(
                head_region,
                helmet["box"]
            )

            if iou > 0.01:
                has_helmet = True
                break

        # ----------------------------------------------------
        # VEST CHECK
        # ----------------------------------------------------

        for vest in vests:

            iou = calculate_iou(
                torso_region,
                vest["box"]
            )

            if iou > 0.01:
                has_vest = True
                break

        # ----------------------------------------------------
        # HAZARD CHECK
        # ----------------------------------------------------

        in_danger_zone = False

        for zone in danger_zones:

            iou = calculate_iou(
                person["box"],
                zone
            )

            if iou > 0.01:
                in_danger_zone = True
                break

        # ====================================================
        # RISK REASONING
        # ====================================================

        if in_danger_zone:

            if not has_helmet and not has_vest:

                risk = "CRITICAL RISK"

                color = CRITICAL_COLOR

            elif not has_helmet:

                risk = "HIGH RISK"

                color = DANGER_COLOR

            elif not has_vest:

                risk = "MEDIUM RISK"

                color = WARNING_COLOR

            else:

                risk = "SAFE IN ZONE"

                color = SAFE_COLOR

        else:

            if has_helmet and has_vest:

                risk = "SAFE"

                color = SAFE_COLOR

            elif not has_helmet and has_vest:

                risk = "NO HELMET"

                color = DANGER_COLOR

            elif has_helmet and not has_vest:

                risk = "NO VEST"

                color = WARNING_COLOR

            else:

                risk = "NO PPE"

                color = DANGER_COLOR

        # ====================================================
        # DRAW PERSON
        # ====================================================

        cv2.rectangle(
            annotated_frame,
            (px1, py1),
            (px2, py2),
            color,
            3
        )

        label = f"Worker {track_id} | {risk}"

        cv2.putText(
            annotated_frame,
            label,
            (px1, py1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

        # ====================================================
        # TRACK TRAJECTORY
        # ====================================================

        center_x = int((px1 + px2) / 2)
        center_y = int((py1 + py2) / 2)

        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append(
            (center_x, center_y)
        )

        if len(track_history[track_id]) > 30:
            track_history[track_id].pop(0)

        points = track_history[track_id]

        for i in range(1, len(points)):

            cv2.line(
                annotated_frame,
                points[i - 1],
                points[i],
                color,
                2
            )

    # ========================================================
    # FPS
    # ========================================================

    current_time = time.time()

    fps_value = 1 / (current_time - prev_time)

    prev_time = current_time

    cv2.putText(
        annotated_frame,
        f"FPS: {fps_value:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # ========================================================
    # FRAME COUNT
    # ========================================================

    frame_count += 1

    cv2.putText(
        annotated_frame,
        f"Frame: {frame_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

    # ========================================================
    # DISPLAY
    # ========================================================

    cv2.imshow(
        "Industrial Hazard Intelligence System",
        annotated_frame
    )

    # ========================================================
    # SAVE FRAME
    # ========================================================

    out.write(annotated_frame)

    # ========================================================
    # EXIT
    # ========================================================

    key = cv2.waitKey(1)

    if key == ord("q"):
        print("\nStopped by user.")
        break

# ============================================================
# RELEASE RESOURCES
# ============================================================

cap.release()

out.release()

cv2.destroyAllWindows()

# ============================================================
# DONE
# ============================================================

print("=" * 60)
print("INDUSTRIAL HAZARD ANALYSIS COMPLETED")
print("=" * 60)

print(f"\nSaved Output:\n{OUTPUT_VIDEO}")

MODEL LOADED SUCCESSFULLY
Resolution : 1920 x 1080
FPS        : 30

Starting industrial hazard intelligence...


Video processing completed.
INDUSTRIAL HAZARD ANALYSIS COMPLETED

Saved Output:
C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\videos\output\04_hazard_intelligence_demo.mp4
